In [17]:
import random
import time
import gc
import joblib
import implicit
import requests
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)

try:
    import spacy
    spacy.load("en_core_web_sm")
except Exception:
    import subprocess, sys, importlib
    print('en_core_web_sm not found — installing into current environment...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'en-core-web-sm==3.8.0'])
    subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])
    importlib.invalidate_caches()
    importlib.reload(__import__('spacy'))
    print('Installation complete — you can now run: nlp = spacy.load("en_core_web_sm")')

try:
    from sklearn.ensemble import RandomForestClassifier as RFC
    from sklearn.metrics import roc_auc_score as roc_auc
except:
    !pip install --quiet scikit-learn
    from sklearn.ensemble import RandomForestClassifier as RFC
    from sklearn.metrics import roc_auc_score as roc_auc

import scipy.sparse as sp
# from statsmodels.stats.proportion import test_proportions_2indep, samplesize_proportions_2indep_onetail
from pandas.api.types import CategoricalDtype

### Task 1: Data Preparation

In [5]:
books = pd.read_csv('./data/books.csv')
ratings = pd.read_csv('./data/ratings.csv')
ratings.info()
ratings.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5976479 entries, 0 to 5976478
Data columns (total 3 columns):
 #   Column   Dtype
---  ------   -----
 0   user_id  int64
 1   book_id  int64
 2   rating   int64
dtypes: int64(3)
memory usage: 136.8 MB


,user_id,book_id,rating
0,1,258,5
1,2,4081,4
2,2,260,5
3,2,9296,5
4,2,2318,3


In [6]:
def preprocess_data(
    ratings: pd.DataFrame,
    train_ratio: float = 0.7

):
    dataset = ratings.copy()

    dataset['implicit_rating'] = dataset.rating \
        .apply(
            lambda x: 1 if x >= 4 else 0
        )

    dataset['book_order'] = dataset \
        .sort_values(['user_id']) \
        .groupby(['user_id']) \
        .cumcount() + 1

    dataset['total_books'] = (
        dataset.groupby('user_id')['book_id'] \
        .transform('count')
    )

    dataset['fraction'] = (
        dataset['book_order']
        / dataset['total_books']
    )

    train = dataset[
        dataset['fraction'] <= train_ratio
    ].copy()

    train['confidence'] = np.log1p(
        train['implicit_rating'] * 10
    )

    test = dataset[
        dataset['fraction'] > train_ratio
    ].copy()

    test['confidence'] = np.log1p(
        test['implicit_rating'] * 10
    )
    
    del dataset
    gc.collect()

    return train, test

In [7]:
train, test = preprocess_data(ratings)
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4159553 entries, 0 to 5976478
Data columns (total 8 columns):
 #   Column           Dtype  
---  ------           -----  
 0   user_id          int64  
 1   book_id          int64  
 2   rating           int64  
 3   implicit_rating  int64  
 4   book_order       int64  
 5   total_books      int64  
 6   fraction         float64
 7   confidence       float64
dtypes: float64(2), int64(6)
memory usage: 285.6 MB


### Task 2: Featuring

In [8]:
def apk(actual, predicted, k=10):

    if len(predicted) > k:
        predicted = predicted[:k]

    score = 0.0
    hits = 0.0

    for i, p in enumerate(predicted):
        if p in actual and p not in predicted[:i]:

            hits += 1.0
            score += hits / (i + 1.0)

    return score / min(len(actual), k)

def mapk(actual_list, predicted_list, k=10):
    scores = []

    for actual, predicted in zip(actual_list, predicted_list):
        # Учитываются пользователи только с положительными оценками в тестовой когорте
        if len(actual) > 0:
            scores.append(
                apk(actual, predicted, k)
            )

    # If no users with actual positive items, return 0 to avoid division by zero
    if not scores:
        return 0.0

    return np.mean(scores)

def simple_ohe(genre_dict, genre):
    if genre in genre_dict: 
        return 1
    return 0

In [9]:
def featuring(train_df, genres_df, books_df):

    train = train_df.copy()
    genres = genres_df.copy()
    books = books_df.copy()

    print('Building genre features...')

    genres = genres[
        genres.book_id.isin(
            books.goodreads_book_id
        )
    ]

    genres.columns = [
        'book_id',
        'genres_dict'
    ]

    all_genres = set()

    for genre_dict in genres.genres_dict:
        all_genres.update(
            genre_dict.keys()
        )

    print(
        f'genres total: {len(all_genres)}'
    )

    for genre in all_genres:

        genres[genre] = genres[
            'genres_dict'
        ].apply(
            lambda x: 1 if genre in x else 0
        ).astype(np.int8)

    genres.drop(
        columns=['genres_dict'],
        inplace=True
    )

    gc.collect()

    print('Building books profile...')

    books_profile = books[
        ['book_id', 'goodreads_book_id']
    ]

    train = train.merge(
        books_profile,
        on='book_id',
        how='left'
    )

    books_profile = books_profile.merge(
        genres,
        left_on='goodreads_book_id',
        right_on='book_id',
        how='left'
    )

    train = train.merge(
        genres,
        left_on='goodreads_book_id',
        right_on='book_id',
        how='left'
    )

    del genres
    gc.collect()

    print('Building user profiles...')

    users_profiles = (
        train.loc[
            train['implicit_rating'] == 1,
            ['user_id'] + list(all_genres)
        ]
        .groupby('user_id')
        .sum()
    )

    users_profiles.columns = [
        f'user_{col}'
        for col in users_profiles.columns
    ]

    joblib.dump(
        users_profiles,
        './joblib/users_profiles.pkl'
    )

    train.columns = [
        f'book_{col}'
        if col in all_genres
        else col
        for col in train.columns
    ]

    books_profile.columns = [
        f'book_{col}'
        if col in all_genres
        else col
        for col in books_profile.columns
    ]

    print('Building title embeddings...')

    nlp = spacy.load(
        "en_core_web_sm"
    )

    books['vector_representation'] = (
        books.title
        .fillna('')
        .apply(lambda x: nlp(x).vector)
    )

    vector_cols = [
        f'vec_component_{i}'
        for i in range(96)
    ]

    vector_df = pd.DataFrame(
        books.vector_representation.tolist(),
        columns=vector_cols,
        index=books.index
    ).astype(np.float32)

    books[vector_cols] = vector_df

    del vector_df

    books.drop(
        columns=['vector_representation'],
        inplace=True
    )

    del nlp
    gc.collect()

    print('Merging embeddings...')

    train = train.merge(
        books[
            vector_cols +
            ['book_id']
        ],
        left_on='book_id_x',
        right_on='book_id'
    )

    books_profile = books_profile.merge(
        books[
            vector_cols +
            ['book_id']
        ],
        left_on='book_id_x',
        right_on='book_id'
    )

    train = train.merge(
        users_profiles,
        left_on='user_id',
        right_index=True,
        how='left'
    )

    del users_profiles
    gc.collect()

    train.fillna(0, inplace=True)

    cols_for_using = (
        [
            f'book_{genre}'
            for genre in all_genres
        ]
        +
        [
            f'user_{genre}'
            for genre in all_genres
        ]
        +
        vector_cols
    )

    print(
        f'RF features: {len(cols_for_using)}'
    )

    print('Training Random Forest...')

    classifier = RFC(
        n_estimators=20,
        criterion='entropy',
        random_state=42,
        n_jobs=-1
    )

    classifier.fit(
        train[cols_for_using],
        train['implicit_rating']
    )

    joblib.dump(
        books_profile,
        './joblib/books_profile.pkl'
    )

    joblib.dump(
        cols_for_using,
        './joblib/cols_for_using.joblib'
    )

    joblib.dump(
        classifier,
        './joblib/rf_classifier.joblib'
    )

    del classifier
    del books_profile

    gc.collect()

    print('Preparing ALS data...')

    als_train = train[
        [
            'user_id',
            'book_id',
            'confidence'
        ]
    ].copy()

    del train
    del books

    gc.collect()

    user_ids = als_train[
        'user_id'
    ].unique()

    book_ids = als_train[
        'book_id'
    ].unique()

    rows = als_train.user_id.astype(
        CategoricalDtype(
            categories=user_ids
        )
    ).cat.codes

    cols = als_train.book_id.astype(
        CategoricalDtype(
            categories=book_ids
        )
    ).cat.codes

    matrix = sp.csr_matrix(
        (
            als_train.confidence,
            (rows, cols)
        ),
        shape=(
            len(user_ids),
            len(book_ids)
        )
    )

    read_history = (
        als_train[
            ['user_id', 'book_id']
        ]
        .groupby('user_id')
        .agg(list)
    )

    read_history_dict = dict(
        zip(
            read_history.index,
            read_history.book_id
        )
    )

    most_popular_book = list(
        ratings.book_id
        .value_counts()
        .head(40)
        .index
    )

    joblib.dump(
        matrix,
        './joblib/user_item_matrix.joblib'
    )

    joblib.dump(
        book_ids,
        './joblib/books_index.joblib'
    )

    joblib.dump(
        user_ids,
        './joblib/user_index.joblib'
    )

    joblib.dump(
        most_popular_book,
        './joblib/most_popular_book.joblib'
    )

    joblib.dump(
        read_history_dict,
        './joblib/read_history_dict.joblib'
    )

    user_index_map = {
        user_id: idx
        for idx, user_id
        in enumerate(user_ids)
    }

    book_index_map = {
        book_id: idx
        for idx, book_id
        in enumerate(book_ids)
    }

    index_user_map = {
        idx: user_id
        for user_id, idx
        in user_index_map.items()
    }

    index_book_map = {
        idx: book_id
        for book_id, idx
        in book_index_map.items()
    }

    del als_train
    gc.collect()

    return (
        matrix,
        user_ids,
        book_ids,
        user_index_map,
        book_index_map,
        index_user_map,
        index_book_map
    )

In [10]:
genres_df = pd.read_json(
    './data/goodreads_book_genres_initial.json',
    lines=True
)

(
    matrix,
    user_ids,
    book_ids,
    user_index_map,
    book_index_map,
    index_user_map,
    index_book_map
) = featuring(train, genres_df, books)

Building genre features...
genres total: 10
Building books profile...
Building user profiles...
Building title embeddings...
Merging embeddings...
RF features: 116
Training Random Forest...
Preparing ALS data...


### Task 3: Modeling

In [11]:
rng = np.random.default_rng(42)

# user_ids = joblib.load(
#     './joblib/user_index.joblib'
# )

sampled_user_ids = rng.choice(
    user_ids,
    size=500,
    replace=False
)

In [12]:
books_profile = joblib.load("./joblib/books_profile.pkl")
users_profiles = joblib.load("./joblib/users_profiles.pkl")
cols_for_using = joblib.load("./joblib/cols_for_using.joblib")
classifier = joblib.load("./joblib/rf_classifier.joblib")

# matrix = joblib.load("./joblib/user_item_matrix.joblib")
# book_ids = joblib.load("./joblib/books_index.joblib")
# user_ids = joblib.load("./joblib/user_index.joblib")

In [13]:
als_model = implicit.als.AlternatingLeastSquares(
    factors=128,
    iterations = 120
)

als_model.fit(sp.csr_matrix(matrix))

/home/yaroslav/anaconda3/envs/notebooks/lib/python3.10/site-packages/implicit/cpu/als.py:96: RuntimeWarning: Intel MKL BLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


  0%|          | 0/120 [00:00<?, ?it/s]

In [31]:
def get_hybrid_recommendations(
    user_id,
    user_index_map,
    als_model,
    matrix,
    book_ids,
    books_profile,
    users_profiles,
    classifier,
    cols_for_using,
    n_als=30,
    n_final=10,
    verbose=False
):

    if user_id not in user_index_map:
        return [], []

    user_idx = user_index_map[user_id]

    als_item_idx, als_scores = als_model.recommend(
        userid=user_idx,
        user_items=matrix[user_idx],
        N=n_als,
        filter_already_liked_items=True
    )

    als_book_ids = [
        int(book_ids[idx])
        for idx in als_item_idx
    ]

    if verbose:

        print("\nALS TOP-30")
        print("-" * 80)

        for rank, (book_id, score) in enumerate(
            zip(als_book_ids, als_scores),
            start=1
        ):
            print(
                f"{rank:02d}. "
                f"book_id={book_id} "
                f"als_score={score:.5f}"
            )

    user_feat = users_profiles.loc[[user_id]]

    reranked = []

    for book_idx, als_score in zip(
        als_item_idx,
        als_scores
    ):

        book_id = int(book_ids[book_idx])

        books_feat = books_profile[
            books_profile.book_id_x == book_id
        ]

        if len(books_feat) == 0:
            continue

        features = user_feat.merge(
            books_feat,
            how="cross"
        )[cols_for_using]

        rf_prob = classifier.predict_proba(
            features
        )[0][1]

        reranked.append(
            (
                book_id,
                rf_prob,
                als_score,
                0.7 * als_score + 0.3 * rf_prob
            )
        )

    reranked = sorted(
        reranked,
        key=lambda x: x[1],
        reverse=True
    )

    hybrid_books = [
        item[0]
        for item in reranked[:n_final]
    ]

    if verbose:

        print("\nHYBRID TOP-10")
        print("-" * 80)

        for rank, (
            book_id,
            rf_prob,
            als_score
        ) in enumerate(
            reranked[:n_final],
            start=1
        ):
            print(
                f"{rank:02d}. "
                f"book_id={book_id} "
                f"rf_prob={rf_prob:.5f} "
                f"als_score={als_score:.5f}"
            )

    return (
        als_book_ids[:n_final],
        hybrid_books
    )

In [32]:
als_predictions = []
hybrid_predictions = []
actual_books_list = []

for user_id in sampled_user_ids:

    actual_books = test[
        (test.user_id == user_id)
        &
        (test.implicit_rating == 1)
    ]["book_id"].tolist()

    if len(actual_books) == 0:
        continue

    als_pred, hybrid_pred = (
        get_hybrid_recommendations(
            user_id=user_id,
            user_index_map=user_index_map,
            als_model=als_model,
            matrix=matrix,
            book_ids=book_ids,
            books_profile=books_profile,
            users_profiles=users_profiles,
            classifier=classifier,
            cols_for_using=cols_for_using,
            n_als=30,
            n_final=10,
            verbose=False
        )
    )

    actual_books_list.append(
        actual_books
    )

    als_predictions.append(
        als_pred
    )

    hybrid_predictions.append(
        hybrid_pred
    )

In [33]:
als_map10 = mapk(
    actual_books_list,
    als_predictions,
    k=10
)

hybrid_map10 = mapk(
    actual_books_list,
    hybrid_predictions,
    k=10
)

print(
    f"ALS mAP@10: {als_map10:.5f}"
)

print(
    f"Hybrid mAP@10: {hybrid_map10:.5f}"
)

print(
    f"Difference: "
    f"{hybrid_map10 - als_map10:.5f}"
)

ALS mAP@10: 0.17346
Hybrid mAP@10: 0.11009
Difference: -0.06338
